# Strict Held-Out Evaluation of Qwen3.5-27B Before and After LoRA

This notebook is evaluation-only.

[实验设置与运行说明](../../../docs/experiment_notes/07_appendix_lora_training/6b9d698c459fd377.md)


In [ ]:
# Run once. Every later code cell reports progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 14 tracked cells.')


## 1. Isolation and frozen evaluation manifest

The evaluator first validates the complete toolkit, adapter manifest, model revision, and Dataset v1 split manifest.


In [ ]:
%%cell_progress 1/14 Mount Drive and select a complete VisualTorque project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Auditing candidate VisualTorque project roots')
from pathlib import Path
import shutil

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
REQUIRED_TOOLKIT_PATHS = (
    'pyproject.toml',
    'src/visual_torque_toolkit/__init__.py',
    'src/visual_torque_toolkit/contracts.py',
    'src/visual_torque_toolkit/classify.py',
    'src/visual_torque_toolkit/dataset.py',
    'src/visual_torque_toolkit/edits.py',
    'src/visual_torque_toolkit/generate.py',
    'src/visual_torque_toolkit/prompts.py',
    'src/visual_torque_toolkit/render.py',
    'src/visual_torque_toolkit/splits.py',
    'src/visual_torque_toolkit/world.py',
    'src/visual_torque_toolkit/behavior/__init__.py',
    'src/visual_torque_toolkit/behavior/adapter.py',
    'src/visual_torque_toolkit/behavior/comparison.py',
    'src/visual_torque_toolkit/behavior/data.py',
    'src/visual_torque_toolkit/behavior/metrics.py',
    'src/visual_torque_toolkit/behavior/parsing.py',
    'src/visual_torque_toolkit/behavior/runner.py',
    'src/visual_torque_toolkit/behavior/schema.py',
)

def missing_toolkit_paths(project_root):
    toolkit_root = project_root / 'toolkit'
    return [
        relative for relative in REQUIRED_TOOLKIT_PATHS
        if not (toolkit_root / relative).is_file()
    ]

candidate_audit = {
    str(candidate): missing_toolkit_paths(candidate)
    for candidate in PROJECT_CANDIDATES
}
complete_candidates = [
    candidate for candidate in PROJECT_CANDIDATES
    if not candidate_audit[str(candidate)]
]
assert complete_candidates, (
    'No complete VisualTorque toolkit was found. Synchronize the whole '
    f'toolkit/ directory. Missing-file audit: {candidate_audit}'
)
PROJECT_ROOT = complete_candidates[0]

cell_stage('Copying toolkit source from Drive to local SSD')
DRIVE_TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
TOOLKIT_ROOT = Path('/content/visualtorque_toolkit_runtime')
if TOOLKIT_ROOT.exists():
    shutil.rmtree(TOOLKIT_ROOT)
shutil.copytree(DRIVE_TOOLKIT_ROOT, TOOLKIT_ROOT)
assert not [
    relative for relative in REQUIRED_TOOLKIT_PATHS
    if not (TOOLKIT_ROOT / relative).is_file()
]
print('Project:', PROJECT_ROOT)
print('Toolkit runtime copy:', TOOLKIT_ROOT)


In [ ]:
%%cell_progress 2/14 Install the frozen evaluation environment
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys

cell_stage('Configuring local model cache before importing Torch')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

cell_stage('Removing unused or incompatible optional packages')
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'uninstall', '-y',
        'torchaudio', 'torchao',
    ],
    check=False,
)
importlib.invalidate_caches()

cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0', 'peft==0.20.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34',
    'safetensors', 'tqdm>=4.66', 'scikit-learn>=1.5',
    'scipy>=1.13', 'pandas>=2.2', 'matplotlib>=3.8',
])

cell_stage('Importing the local immutable toolkit copy')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import visual_torque_toolkit
from IPython.display import display
from peft import PeftModel
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, f1_score
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.behavior.adapter import (
    HuggingFaceMultimodalAdapter,
)
from visual_torque_toolkit.behavior.parsing import parse_choice_label

# Defensive compatibility patch: Qwen3.5 requires mm_token_type_ids to grow
# with any literal text tokens appended for exact candidate scoring.
def append_text_prefix_aligned(self, inputs, prefix):
    if not prefix:
        return inputs
    batch_size = inputs['input_ids'].shape[0]
    prefix_tensor = torch.tensor(
        prefix,
        dtype=inputs['input_ids'].dtype,
        device=inputs['input_ids'].device,
    ).unsqueeze(0).expand(batch_size, -1)
    inputs['input_ids'] = torch.cat(
        (inputs['input_ids'], prefix_tensor), dim=1
    )
    attention_extension = torch.ones(
        (batch_size, len(prefix)),
        dtype=inputs['attention_mask'].dtype,
        device=inputs['attention_mask'].device,
    )
    inputs['attention_mask'] = torch.cat(
        (inputs['attention_mask'], attention_extension), dim=1
    )
    for key in ('token_type_ids', 'mm_token_type_ids'):
        token_types = inputs.get(key)
        if token_types is not None and token_types.ndim == 2:
            text_extension = torch.zeros(
                (batch_size, len(prefix)),
                dtype=token_types.dtype,
                device=token_types.device,
            )
            inputs[key] = torch.cat((token_types, text_extension), dim=1)
    inputs.pop('position_ids', None)
    return inputs

HuggingFaceMultimodalAdapter._append_text_prefix = append_text_prefix_aligned
torch.backends.cuda.matmul.allow_tf32 = True
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents
try:
    torchao_version = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
assert torchao_version is None
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('transformers:', importlib.metadata.version('transformers'))
print('peft:', importlib.metadata.version('peft'))
print('Toolkit:', toolkit_path)


In [ ]:
%%cell_progress 3/14 Lock model, adapter, evaluation, and hardware identities
cell_stage('Checking the 96GB single-GPU runtime')
assert torch.cuda.is_available(), 'Select a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
assert GPU_MEMORY_GIB >= 88, (GPU_NAME, GPU_MEMORY_GIB)
if GPU_CAPABILITY[0] >= 10:
    cuda_tuple = tuple(int(value) for value in torch.version.cuda.split('.')[:2])
    assert cuda_tuple >= (12, 8), (torch.__version__, torch.version.cuda)

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
EXPECTED_LAYER_COUNT = 64
SEED = 20260903
CANDIDATE_BATCH_SIZE = 2
GENERATION_BATCH_SIZE = 2
BOOTSTRAP_REPLICATES = 10000
REUSE_VERIFIED_TRAINING_RUN_CANDIDATE_SHARDS = True

TRAIN_RUN_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_bf16_lora_weight_shortcut_v1'
)
ADAPTER_DIR = TRAIN_RUN_ROOT / 'adapter_final'
ADAPTER_MANIFEST_PATH = TRAIN_RUN_ROOT / 'ADAPTER_MANIFEST.json'
PROTOCOL_PATH = TRAIN_RUN_ROOT / 'frozen_protocol.json'
assert ADAPTER_MANIFEST_PATH.is_file(), ADAPTER_MANIFEST_PATH
assert PROTOCOL_PATH.is_file(), PROTOCOL_PATH
assert (ADAPTER_DIR / 'adapter_config.json').is_file(), ADAPTER_DIR

adapter_manifest = json.loads(
    ADAPTER_MANIFEST_PATH.read_text(encoding='utf-8')
)
training_protocol = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
assert adapter_manifest['base_model_id'] == MODEL_ID
assert adapter_manifest['base_model_revision'] == MODEL_REVISION
assert training_protocol['identity']['train_source_split'] == 'probe_train'
assert training_protocol['identity']['final_test_split'] == 'probe_test'

EVAL_VERSION = 'v1'
OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / f'qwen27b_postlora_heldout_evaluation_{EVAL_VERSION}'
)
BASE_CANDIDATE_DIR = OUTPUT_ROOT / 'base_candidate_shards'
LORA_CANDIDATE_DIR = OUTPUT_ROOT / 'lora_candidate_shards'
BASE_GENERATION_DIR = OUTPUT_ROOT / 'base_generation_shards'
LORA_GENERATION_DIR = OUTPUT_ROOT / 'lora_generation_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for directory in (
    OUTPUT_ROOT, BASE_CANDIDATE_DIR, LORA_CANDIDATE_DIR,
    BASE_GENERATION_DIR, LORA_GENERATION_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print('GPU:', GPU_NAME, '| memory GiB:', round(GPU_MEMORY_GIB, 1))
print('Frozen adapter:', ADAPTER_DIR)
print('Evaluation output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 4/14 Prepare the source Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
SOURCE_DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
DRIVE_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(dataset_path):
    return dataset_path.is_dir() and all(
        (dataset_path / filename).is_file()
        for filename in REQUIRED_DATASET_FILES
    )

if not dataset_complete(SOURCE_DATASET_DIR):
    SOURCE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [member for member in bundle.infolist() if not member.is_dir()]
            extraction_root = SOURCE_DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (SOURCE_DATASET_DIR / member.filename).resolve()
                if extraction_root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, SOURCE_DATASET_DIR)
        if not dataset_complete(SOURCE_DATASET_DIR):
            nested = [
                item for item in SOURCE_DATASET_DIR.iterdir()
                if dataset_complete(item)
            ]
            assert len(nested) == 1, nested
            SOURCE_DATASET_DIR = nested[0]
    elif dataset_complete(DRIVE_DATASET_DIR):
        cell_stage('Copying Dataset v1 from Drive')
        shutil.copytree(
            DRIVE_DATASET_DIR, SOURCE_DATASET_DIR, dirs_exist_ok=True
        )
    else:
        raise FileNotFoundError(
            'Place dataset_v1.zip or the full dataset_v1 directory under '
            'VisualTorque/data on Drive.'
        )
assert dataset_complete(SOURCE_DATASET_DIR)
print('Source dataset:', SOURCE_DATASET_DIR)


In [ ]:
%%cell_progress 5/14 Materialize and audit a physically test-only dataset
cell_stage('Streaming only probe_test records into an isolated directory')
import hashlib
import math

def read_jsonl(file_path):
    with Path(file_path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_json_atomic(file_path, value):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(file_path)

def write_jsonl_atomic(file_path, rows):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + '\n')
    temporary.replace(file_path)

def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

split_record = json.loads(
    (SOURCE_DATASET_DIR / 'splits.json').read_text(encoding='utf-8')
)
frozen_train_ids = set(split_record['probe_scenes']['probe_train'])
frozen_dev_ids = set(split_record['probe_scenes']['probe_dev'])
frozen_test_ids = set(split_record['probe_scenes']['probe_test'])
assert not (frozen_train_ids & frozen_test_ids)
assert not (frozen_dev_ids & frozen_test_ids)

test_scenes = [
    row for row in read_jsonl(SOURCE_DATASET_DIR / 'scenes.jsonl')
    if row['split'] == 'probe_test'
]
test_questions = [
    row for row in read_jsonl(SOURCE_DATASET_DIR / 'questions.jsonl')
    if row['split'] == 'probe_test' and row['question_kind'] == 'outcome'
]
test_scene_ids = {row['scene_id'] for row in test_scenes}
assert test_scene_ids == frozen_test_ids
assert all(row['scene_id'] in frozen_test_ids for row in test_questions)
assert all(row['scene_id'] not in frozen_train_ids for row in test_questions)
assert all(row['scene_id'] not in frozen_dev_ids for row in test_questions)

TEST_ONLY_DATASET_DIR = Path('/content/visualtorque_probe_test_only')
if TEST_ONLY_DATASET_DIR.exists():
    shutil.rmtree(TEST_ONLY_DATASET_DIR)
(TEST_ONLY_DATASET_DIR / 'images').mkdir(parents=True)
write_jsonl_atomic(TEST_ONLY_DATASET_DIR / 'scenes.jsonl', test_scenes)
write_jsonl_atomic(TEST_ONLY_DATASET_DIR / 'questions.jsonl', test_questions)

cell_stage('Hard-linking only held-out images into the evaluator')
for scene in tqdm(test_scenes, desc='Isolating test images', unit='image'):
    source = SOURCE_DATASET_DIR / scene['image_path']
    target = TEST_ONLY_DATASET_DIR / scene['image_path']
    target.parent.mkdir(parents=True, exist_ok=True)
    assert source.is_file(), source
    try:
        os.link(source, target)
    except OSError:
        shutil.copy2(source, target)

source_hashes = {
    name: sha256_file(SOURCE_DATASET_DIR / name)
    for name in sorted(REQUIRED_DATASET_FILES)
}
test_only_manifest = {
    'schema_version': 'visualtorque-test-only-evaluator-1.0',
    'allowed_split': 'probe_test',
    'forbidden_splits': ['probe_train', 'probe_dev'],
    'scene_count': len(test_scenes),
    'outcome_question_count': len(test_questions),
    'scene_ids_sha256': 'sha256:' + hashlib.sha256(
        '\n'.join(sorted(test_scene_ids)).encode('utf-8')
    ).hexdigest(),
    'source_dataset_hashes': source_hashes,
}
write_json_atomic(
    OUTPUT_ROOT / 'test_only_dataset_manifest.json', test_only_manifest
)
print('Isolated scenes:', len(test_scenes))
print('Isolated outcome prompts:', len(test_questions))
print('No train/dev records or images are present in:', TEST_ONLY_DATASET_DIR)


In [ ]:
%%cell_progress 6/14 Build frozen candidate and deterministic-generation manifests
cell_stage('Building examples from the physically test-only directory')
test_config = BehaviorRunConfig(
    dataset_dir=TEST_ONLY_DATASET_DIR,
    output_dir=OUTPUT_ROOT / '_unused_behavior_config',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_test',),
    batch_size=CANDIDATE_BATCH_SIZE,
    seed=SEED,
)
candidate_examples = list(load_behavior_examples(test_config))
assert candidate_examples
assert all(example.split == 'probe_test' for example in candidate_examples)
assert all(example.scene_id in frozen_test_ids for example in candidate_examples)
assert not any(example.scene_id in frozen_train_ids for example in candidate_examples)
assert not any(example.scene_id in frozen_dev_ids for example in candidate_examples)

examples_by_scene = {}
for example in candidate_examples:
    examples_by_scene.setdefault(example.scene_id, []).append(example)
assert set(examples_by_scene) == frozen_test_ids
assert all(len(rows) == 6 for rows in examples_by_scene.values())

scene_by_id = {row['scene_id']: row for row in test_scenes}
SIDES = ('left', 'balanced', 'right')
LABELS = ('A', 'B', 'C')

def side_from_comparison(left_value, right_value):
    if left_value > right_value:
        return 'left'
    if left_value < right_value:
        return 'right'
    return 'balanced'

def weight_cue(scene):
    return side_from_comparison(
        int(scene['derived']['left_units']),
        int(scene['derived']['right_units']),
    )

def shortcut_relation(scene):
    cue = weight_cue(scene)
    outcome = scene['derived']['outcome']
    if cue == 'balanced':
        return 'weight_neutral'
    if cue == outcome:
        return 'weight_aligned'
    return 'weight_conflict'

scene_rows = [{
    'scene_id': scene['scene_id'],
    'outcome': scene['derived']['outcome'],
    'weight_cue': weight_cue(scene),
    'shortcut_relation': shortcut_relation(scene),
    'problem_type': scene['problem_type'],
} for scene in test_scenes]
scene_metadata = pd.DataFrame(scene_rows)

cell_stage('Freezing the 3x3 no-replacement shortcut subset')
balance_cells = [(cue, outcome) for cue in SIDES for outcome in SIDES]
grouped = {}
for cue, outcome in balance_cells:
    current = scene_metadata[
        (scene_metadata['weight_cue'] == cue)
        & (scene_metadata['outcome'] == outcome)
    ].copy()
    current = current.sort_values(
        'scene_id', key=lambda values: values.map(stable_key)
    )
    assert len(current) > 0, (cue, outcome)
    grouped[(cue, outcome)] = current['scene_id'].tolist()
balanced_per_cell = min(len(ids) for ids in grouped.values())
balanced_test_ids = {
    scene_id
    for cell in balance_cells
    for scene_id in grouped[cell][:balanced_per_cell]
}
assert len(balanced_test_ids) == balanced_per_cell * 9

cell_stage('Selecting one globally label-balanced generation prompt per scene')
ordered_scene_ids = sorted(frozen_test_ids, key=stable_key)
generation_one_per_scene = []
for index, scene_id in enumerate(ordered_scene_ids):
    desired_label = LABELS[index % len(LABELS)]
    candidates = sorted(
        (
            example for example in examples_by_scene[scene_id]
            if example.gold_label == desired_label
        ),
        key=lambda example: stable_key(example.question_id),
    )
    assert candidates, (scene_id, desired_label)
    generation_one_per_scene.append(candidates[0])

generation_balanced_six = [
    example for example in candidate_examples
    if example.scene_id in balanced_test_ids
]
generation_example_by_id = {
    example.example_id: example
    for example in generation_one_per_scene + generation_balanced_six
}
generation_examples = list(generation_example_by_id.values())
generation_full_ids = {example.example_id for example in generation_one_per_scene}
generation_balanced_six_ids = {
    example.example_id for example in generation_balanced_six
}

label_counts = pd.Series(
    [example.gold_label for example in generation_one_per_scene]
).value_counts().reindex(LABELS, fill_value=0)
assert label_counts.max() - label_counts.min() <= 1
assert all(example.split == 'probe_test' for example in generation_examples)

evaluation_manifest = {
    'schema_version': 'visualtorque-heldout-evaluation-manifest-1.0',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_manifest_sha256': sha256_file(ADAPTER_MANIFEST_PATH),
    'allowed_model_input_split': 'probe_test',
    'candidate_scene_ids': sorted(frozen_test_ids),
    'candidate_example_ids': sorted(
        example.example_id for example in candidate_examples
    ),
    'balanced_shortcut_scene_ids': sorted(balanced_test_ids),
    'generation_one_per_scene_example_ids': sorted(generation_full_ids),
    'generation_balanced_six_example_ids': sorted(
        generation_balanced_six_ids
    ),
    'seed': SEED,
}
EVALUATION_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_evaluation_manifest.json'
if EVALUATION_MANIFEST_PATH.is_file():
    previous = json.loads(EVALUATION_MANIFEST_PATH.read_text(encoding='utf-8'))
    assert previous == evaluation_manifest, (
        'Evaluation manifest changed. Increment EVAL_VERSION.'
    )
else:
    write_json_atomic(EVALUATION_MANIFEST_PATH, evaluation_manifest)

display(pd.crosstab(
    scene_metadata[scene_metadata['scene_id'].isin(balanced_test_ids)]['weight_cue'],
    scene_metadata[scene_metadata['scene_id'].isin(balanced_test_ids)]['outcome'],
).reindex(index=SIDES, columns=SIDES, fill_value=0))
display(label_counts.rename('generation_prompt_count').to_frame())
print('Candidate examples:', len(candidate_examples))
print('Generation examples to execute per model:', len(generation_examples))
print('Frozen manifest:', EVALUATION_MANIFEST_PATH)


## 2. Native evaluation before attaching LoRA

The exact pinned base checkpoint is loaded in inference-only mode.


In [ ]:
%%cell_progress 7/14 Load and freeze the native Qwen3.5-27B model
cell_stage('Loading the pinned non-quantized BF16 checkpoint')
import gc

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map='auto',
)
assert adapter.metadata['resolved_revision'] == MODEL_REVISION
quantization_config = getattr(adapter.model.config, 'quantization_config', None)
assert quantization_config is None, quantization_config
device_map = getattr(adapter.model, 'hf_device_map', {}) or {}
assert not {
    name: device for name, device in device_map.items()
    if str(device).lower() in {'cpu', 'disk'}
}
for parameter in adapter.model.parameters():
    parameter.requires_grad_(False)
adapter.model.eval()
assert not any(parameter.requires_grad for parameter in adapter.model.parameters())

def resolve_module_path(root, module_path):
    value = root
    for component in module_path.split('.'):
        value = getattr(value, component)
    return value

layer_paths = (
    'model.language_model.layers', 'model.text_model.layers',
    'language_model.model.layers', 'model.model.layers', 'model.layers',
)
decoder_layers = None
for layer_path in layer_paths:
    try:
        candidate = resolve_module_path(adapter.model, layer_path)
    except AttributeError:
        continue
    if isinstance(candidate, torch.nn.ModuleList):
        decoder_layers = candidate
        break
assert decoder_layers is not None and len(decoder_layers) == EXPECTED_LAYER_COUNT
print('Native model loaded with', len(decoder_layers), 'decoder layers.')
print('CUDA allocated GiB:', round(torch.cuda.memory_allocated() / 2**30, 2))


In [ ]:
%%cell_progress 8/14 Verify or compute native candidate-scored predictions
cell_stage('Defining identity-checked resumable candidate scoring')

def shard_path(shard_dir, example_id):
    digest = hashlib.sha256(example_id.encode('utf-8')).hexdigest()[:24]
    return Path(shard_dir) / f'{digest}.json'

def normalize_candidate_row(row, example, run_name):
    scene = scene_by_id[example.scene_id]
    semantic_scores = {
        semantic: float(row['logprob_by_semantic'][semantic])
        for semantic in SIDES
    }
    return {
        'run_name': run_name,
        'example_id': example.example_id,
        'question_id': example.question_id,
        'scene_id': example.scene_id,
        'variant': example.variant,
        'problem_type': example.problem_type,
        'outcome': example.outcome,
        'weight_cue': weight_cue(scene),
        'shortcut_relation': shortcut_relation(scene),
        'balanced_shortcut_test': example.scene_id in balanced_test_ids,
        'logprob_by_semantic': semantic_scores,
        'scoring_mode': row['scoring_mode'],
    }

def load_complete_candidate_cache(shard_dir, examples, accepted_run_name):
    shard_dir = Path(shard_dir)
    expected = {example.example_id: example for example in examples}
    rows = []
    for example_id, example in expected.items():
        path = shard_path(shard_dir, example_id)
        if not path.is_file():
            return None
        row = json.loads(path.read_text(encoding='utf-8'))
        if row.get('example_id') != example_id:
            return None
        if row.get('run_name') != accepted_run_name:
            return None
        if set(row.get('logprob_by_semantic', {})) != set(SIDES):
            return None
        rows.append(normalize_candidate_row(row, example, accepted_run_name))
    return rows

def evaluate_candidates(examples, shard_dir, run_name):
    shard_dir = Path(shard_dir)
    pending = [
        example for example in examples
        if not shard_path(shard_dir, example.example_id).is_file()
    ]
    adapter.model.eval()
    for start in tqdm(
        range(0, len(pending), CANDIDATE_BATCH_SIZE),
        desc=f'{run_name} candidates', unit='batch',
    ):
        batch = pending[start:start + CANDIDATE_BATCH_SIZE]
        results = adapter.score_batch(batch)
        for example, result in zip(batch, results):
            semantic_scores = {
                str(example.label_to_semantic[label]): float(score)
                for label, score in result.logprob_by_label.items()
            }
            payload = {
                'run_name': run_name,
                'example_id': example.example_id,
                'question_id': example.question_id,
                'scene_id': example.scene_id,
                'variant': example.variant,
                'logprob_by_semantic': semantic_scores,
                'scoring_mode': result.scoring_mode,
            }
            write_json_atomic(
                shard_path(shard_dir, example.example_id), payload
            )
    rows = load_complete_candidate_cache(shard_dir, examples, run_name)
    assert rows is not None
    return rows

cached_base_dir = TRAIN_RUN_ROOT / 'baseline_eval_shards'
base_candidate_rows = None
if REUSE_VERIFIED_TRAINING_RUN_CANDIDATE_SHARDS:
    base_candidate_rows = load_complete_candidate_cache(
        cached_base_dir, candidate_examples, 'bf16_base'
    )
if base_candidate_rows is not None:
    print('Reused', len(base_candidate_rows), 'verified native candidate shards.')
else:
    cell_stage('Computing missing native candidate scores')
    base_candidate_rows = evaluate_candidates(
        candidate_examples, BASE_CANDIDATE_DIR, 'native_confirmatory'
    )
write_jsonl_atomic(
    OUTPUT_ROOT / 'native_candidate_records.jsonl', base_candidate_rows
)


In [ ]:
%%cell_progress 9/14 Run deterministic native answer generation
cell_stage('Defining resumable exact-answer generation')

def normalize_generation_row(row, example, run_name):
    parsed_label = row.get('parsed_label')
    strict_label = row.get('strict_label')
    predicted_semantic = (
        str(example.label_to_semantic[parsed_label])
        if parsed_label in example.labels else None
    )
    return {
        'run_name': run_name,
        'example_id': example.example_id,
        'question_id': example.question_id,
        'scene_id': example.scene_id,
        'variant': example.variant,
        'gold_label': example.gold_label,
        'gold_semantic': str(example.gold_semantic),
        'parsed_label': parsed_label,
        'strict_label': strict_label,
        'predicted_semantic': predicted_semantic,
        'parsed_correct': parsed_label == example.gold_label,
        'strict_exact_correct': strict_label == example.gold_label,
        'valid_answer': parsed_label in example.labels,
        'strict_one_letter_format': strict_label in example.labels,
        'parsed_content': row.get('parsed_content', ''),
        'raw_response': row.get('raw_response', ''),
    }

def load_complete_generation_cache(shard_dir, examples, run_name):
    expected = {example.example_id: example for example in examples}
    rows = []
    for example_id, example in expected.items():
        path = shard_path(shard_dir, example_id)
        if not path.is_file():
            return None
        row = json.loads(path.read_text(encoding='utf-8'))
        if row.get('example_id') != example_id or row.get('run_name') != run_name:
            return None
        rows.append(normalize_generation_row(row, example, run_name))
    return rows

def evaluate_generation(examples, shard_dir, run_name):
    shard_dir = Path(shard_dir)
    shard_dir.mkdir(parents=True, exist_ok=True)
    pending = [
        example for example in examples
        if not shard_path(shard_dir, example.example_id).is_file()
    ]
    adapter.model.eval()
    for start in tqdm(
        range(0, len(pending), GENERATION_BATCH_SIZE),
        desc=f'{run_name} generation', unit='batch',
    ):
        batch = pending[start:start + GENERATION_BATCH_SIZE]
        results = adapter.generate_batch(batch)
        for example, result in zip(batch, results):
            content = result.parsed_content.strip()
            strict_label = content if content in example.labels else None
            parsed_label = parse_choice_label(content, example.labels)
            payload = {
                'run_name': run_name,
                'example_id': example.example_id,
                'question_id': example.question_id,
                'scene_id': example.scene_id,
                'variant': example.variant,
                'gold_label': example.gold_label,
                'gold_semantic': str(example.gold_semantic),
                'parsed_label': parsed_label,
                'strict_label': strict_label,
                'parsed_content': content,
                'raw_response': result.raw_response,
            }
            write_json_atomic(
                shard_path(shard_dir, example.example_id), payload
            )
    rows = load_complete_generation_cache(shard_dir, examples, run_name)
    assert rows is not None
    return rows

base_generation_rows = evaluate_generation(
    generation_examples, BASE_GENERATION_DIR, 'native_generation'
)
write_jsonl_atomic(
    OUTPUT_ROOT / 'native_generation_records.jsonl', base_generation_rows
)
print('Native generations:', len(base_generation_rows))


## 3. Attach the frozen adapter and repeat identically

Every saved adapter file is checked against the hash recorded when training completed.


In [ ]:
%%cell_progress 10/14 Load the frozen LoRA adapter without training
cell_stage('Verifying every saved adapter artifact')
for relative_path, expected_hash in adapter_manifest['adapter_files'].items():
    current_path = ADAPTER_DIR / relative_path
    assert current_path.is_file(), current_path
    assert sha256_file(current_path) == expected_hash, (
        'Adapter file changed after training.', relative_path
    )

cell_stage('Attaching the saved adapter in inference-only mode')
adapter.model = PeftModel.from_pretrained(
    adapter.model,
    str(ADAPTER_DIR),
    is_trainable=False,
)
for parameter in adapter.model.parameters():
    parameter.requires_grad_(False)
adapter.model.eval()
assert not any(parameter.requires_grad for parameter in adapter.model.parameters())
assert not adapter.model.training
print('Frozen LoRA attached; trainable parameters: 0')
print('CUDA allocated GiB:', round(torch.cuda.memory_allocated() / 2**30, 2))


In [ ]:
%%cell_progress 11/14 Verify/compute LoRA candidates and generate answers
cell_stage('Verifying or computing LoRA candidate scores')
cached_lora_dir = TRAIN_RUN_ROOT / 'adapted_eval_shards'
lora_candidate_rows = None
if REUSE_VERIFIED_TRAINING_RUN_CANDIDATE_SHARDS:
    lora_candidate_rows = load_complete_candidate_cache(
        cached_lora_dir, candidate_examples, 'bf16_lora'
    )
if lora_candidate_rows is not None:
    print('Reused', len(lora_candidate_rows), 'verified LoRA candidate shards.')
else:
    lora_candidate_rows = evaluate_candidates(
        candidate_examples, LORA_CANDIDATE_DIR, 'lora_confirmatory'
    )
write_jsonl_atomic(
    OUTPUT_ROOT / 'lora_candidate_records.jsonl', lora_candidate_rows
)

cell_stage('Running deterministic LoRA answer generation')
lora_generation_rows = evaluate_generation(
    generation_examples, LORA_GENERATION_DIR, 'lora_generation'
)
write_jsonl_atomic(
    OUTPUT_ROOT / 'lora_generation_records.jsonl', lora_generation_rows
)
print('LoRA generations:', len(lora_generation_rows))


## 4. Paired analysis and handoff

All headline comparisons are paired by scene.


In [ ]:
%%cell_progress 12/14 Analyze held-out candidate accuracy and shortcut behavior
cell_stage('Aggregating all six mappings into semantic scene predictions')

def aggregate_candidate_rows(rows, model_name):
    question_frame = pd.DataFrame(rows)
    question_outputs = []
    scene_outputs = []
    for scene_id, current in question_frame.groupby('scene_id'):
        assert len(current) == 6, (scene_id, len(current))
        per_question_predictions = []
        for row in current.itertuples(index=False):
            prediction = max(
                SIDES,
                key=lambda semantic: row.logprob_by_semantic[semantic],
            )
            per_question_predictions.append(prediction)
            question_outputs.append({
                'model': model_name,
                'example_id': row.example_id,
                'scene_id': scene_id,
                'variant': row.variant,
                'prediction': prediction,
                'outcome': row.outcome,
                'correct': prediction == row.outcome,
            })
        mean_scores = {
            semantic: float(np.mean([
                scores[semantic]
                for scores in current['logprob_by_semantic']
            ]))
            for semantic in SIDES
        }
        prediction = max(SIDES, key=lambda semantic: mean_scores[semantic])
        score_values = np.asarray([mean_scores[side] for side in SIDES])
        probabilities = np.exp(score_values - score_values.max())
        probabilities /= probabilities.sum()
        first = current.iloc[0]
        gold_index = SIDES.index(first['outcome'])
        scene_outputs.append({
            'model': model_name,
            'scene_id': scene_id,
            'problem_type': first['problem_type'],
            'outcome': first['outcome'],
            'weight_cue': first['weight_cue'],
            'shortcut_relation': first['shortcut_relation'],
            'balanced_shortcut_test': bool(first['balanced_shortcut_test']),
            'prediction': prediction,
            'correct': prediction == first['outcome'],
            'follows_weight': prediction == first['weight_cue'],
            'confidence': float(probabilities.max()),
            'gold_nll': float(-np.log(max(probabilities[gold_index], 1e-12))),
            'unique_permutation_predictions': len(set(per_question_predictions)),
            'permutation_unanimous': len(set(per_question_predictions)) == 1,
            **{f'score_{side}': mean_scores[side] for side in SIDES},
        })
    return (
        pd.DataFrame(scene_outputs).sort_values('scene_id').reset_index(drop=True),
        pd.DataFrame(question_outputs),
    )

base_scene, base_question = aggregate_candidate_rows(
    base_candidate_rows, 'native'
)
lora_scene, lora_question = aggregate_candidate_rows(
    lora_candidate_rows, 'lora'
)
assert base_scene['scene_id'].tolist() == lora_scene['scene_id'].tolist()

def subset_masks(frame):
    return {
        'all_probe_test': pd.Series(True, index=frame.index),
        'balanced_shortcut_test': frame['balanced_shortcut_test'],
        'all_weight_conflict': frame['shortcut_relation'] == 'weight_conflict',
        'balanced_weight_conflict': (
            frame['balanced_shortcut_test']
            & (frame['shortcut_relation'] == 'weight_conflict')
        ),
        'weight_aligned': frame['shortcut_relation'] == 'weight_aligned',
        'weight_neutral': frame['shortcut_relation'] == 'weight_neutral',
        'multi_stack': frame['problem_type'] == 'multi_stack',
        'non_multi_stack': frame['problem_type'] != 'multi_stack',
    }

def candidate_metric_row(frame, model_name, subset_name):
    recalls = []
    for outcome in SIDES:
        current = frame[frame['outcome'] == outcome]
        recalls.append(float(current['correct'].mean()) if len(current) else np.nan)
    informative = frame['weight_cue'] != 'balanced'
    return {
        'model': model_name,
        'subset': subset_name,
        'scenes': int(len(frame)),
        'accuracy': float(frame['correct'].mean()),
        'balanced_accuracy': float(np.nanmean(recalls)),
        'macro_f1': float(f1_score(
            frame['outcome'], frame['prediction'], labels=SIDES,
            average='macro', zero_division=0,
        )),
        'left_recall': recalls[0],
        'balanced_recall': recalls[1],
        'right_recall': recalls[2],
        'weight_follow_rate': float(
            frame.loc[informative, 'follows_weight'].mean()
        ),
        'permutation_unanimous_rate': float(
            frame['permutation_unanimous'].mean()
        ),
        'mean_gold_nll': float(frame['gold_nll'].mean()),
    }

candidate_summary_rows = []
for model_name, frame in (('native', base_scene), ('lora', lora_scene)):
    for subset_name, mask in subset_masks(frame).items():
        candidate_summary_rows.append(candidate_metric_row(
            frame[mask], model_name, subset_name
        ))
candidate_summary = pd.DataFrame(candidate_summary_rows)
candidate_summary.to_csv(
    ANALYSIS_DIR / 'candidate_accuracy_by_subset.csv', index=False
)
base_scene.to_csv(ANALYSIS_DIR / 'native_scene_predictions.csv', index=False)
lora_scene.to_csv(ANALYSIS_DIR / 'lora_scene_predictions.csv', index=False)

paired = base_scene.merge(
    lora_scene,
    on='scene_id', suffixes=('_native', '_lora'), validate='one_to_one',
)
paired['delta_correct'] = (
    paired['correct_lora'].astype(float)
    - paired['correct_native'].astype(float)
)
paired['delta_follows_weight'] = (
    paired['follows_weight_lora'].astype(float)
    - paired['follows_weight_native'].astype(float)
)
paired.to_csv(ANALYSIS_DIR / 'paired_candidate_scenes.csv', index=False)

def paired_bootstrap(values, key):
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(
        int(stable_key(f'bootstrap:{key}')[:16], 16) % (2**63 - 1)
    )
    draws = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
    for start in range(0, BOOTSTRAP_REPLICATES, 1000):
        count = min(1000, BOOTSTRAP_REPLICATES - start)
        indexes = generator.integers(
            0, len(values), size=(count, len(values))
        )
        draws[start:start + count] = values[indexes].mean(axis=1)
    return {
        'mean': float(values.mean()),
        'ci_low': float(np.quantile(draws, 0.025)),
        'ci_high': float(np.quantile(draws, 0.975)),
    }

def paired_sign_test(frame):
    improved = int((~frame['correct_native'] & frame['correct_lora']).sum())
    harmed = int((frame['correct_native'] & ~frame['correct_lora']).sum())
    discordant = improved + harmed
    p_value = (
        float(binomtest(improved, discordant, p=0.5).pvalue)
        if discordant else 1.0
    )
    return {
        'native_wrong_lora_right': improved,
        'native_right_lora_wrong': harmed,
        'discordant_scenes': discordant,
        'exact_mcnemar_p': p_value,
    }

paired_masks = {
    'all_probe_test': pd.Series(True, index=paired.index),
    'balanced_shortcut_test': paired['balanced_shortcut_test_native'],
    'all_weight_conflict': (
        paired['shortcut_relation_native'] == 'weight_conflict'
    ),
    'balanced_weight_conflict': (
        paired['balanced_shortcut_test_native']
        & (paired['shortcut_relation_native'] == 'weight_conflict')
    ),
}
paired_effects = {}
for name, mask in paired_masks.items():
    current = paired[mask]
    paired_effects[name] = {
        'accuracy_change': paired_bootstrap(
            current['delta_correct'], f'{name}:accuracy'
        ),
        'mcnemar': paired_sign_test(current),
    }
    informative = current['weight_cue_native'] != 'balanced'
    if informative.any():
        paired_effects[name]['weight_follow_change'] = paired_bootstrap(
            current.loc[informative, 'delta_follows_weight'],
            f'{name}:weight-follow',
        )
write_json_atomic(ANALYSIS_DIR / 'paired_candidate_effects.json', paired_effects)

headline = candidate_summary[
    candidate_summary['subset'].isin((
        'all_probe_test', 'balanced_shortcut_test',
        'all_weight_conflict', 'balanced_weight_conflict',
    ))
].pivot(index='subset', columns='model', values=[
    'scenes', 'accuracy', 'balanced_accuracy', 'macro_f1',
    'balanced_recall', 'weight_follow_rate',
    'permutation_unanimous_rate',
])
display(headline.round(4))
display(pd.DataFrame({
    name: {
        'accuracy_delta': result['accuracy_change']['mean'],
        'ci_low': result['accuracy_change']['ci_low'],
        'ci_high': result['accuracy_change']['ci_high'],
        'mcnemar_p': result['mcnemar']['exact_mcnemar_p'],
    }
    for name, result in paired_effects.items()
}).T.round(6))


In [ ]:
%%cell_progress 13/14 Analyze generated answers and create final figures
cell_stage('Comparing deterministic generated answers')
base_generation = pd.DataFrame(base_generation_rows)
lora_generation = pd.DataFrame(lora_generation_rows)

def generation_metric(frame, ids, model_name, scope_name):
    current = frame[frame['example_id'].isin(ids)].copy()
    assert len(current) == len(ids), (scope_name, len(current), len(ids))
    by_scene = current.groupby('scene_id').agg(
        parsed_accuracy=('parsed_correct', 'mean'),
        strict_accuracy=('strict_exact_correct', 'mean'),
        valid_rate=('valid_answer', 'mean'),
        strict_format_rate=('strict_one_letter_format', 'mean'),
        semantic_predictions=(
            'predicted_semantic',
            lambda values: len(set(value for value in values if value is not None)),
        ),
    ).reset_index()
    return {
        'model': model_name,
        'scope': scope_name,
        'scenes': int(by_scene['scene_id'].nunique()),
        'prompts': int(len(current)),
        'parsed_answer_accuracy': float(by_scene['parsed_accuracy'].mean()),
        'strict_exact_match_accuracy': float(by_scene['strict_accuracy'].mean()),
        'valid_answer_rate': float(by_scene['valid_rate'].mean()),
        'strict_one_letter_format_rate': float(
            by_scene['strict_format_rate'].mean()
        ),
        'semantic_unanimity_rate': float(
            (by_scene['semantic_predictions'] == 1).mean()
        ),
    }

generation_summary = pd.DataFrame([
    generation_metric(
        base_generation, generation_full_ids,
        'native', 'one_balanced-label_prompt_per_test_scene',
    ),
    generation_metric(
        lora_generation, generation_full_ids,
        'lora', 'one_balanced-label_prompt_per_test_scene',
    ),
    generation_metric(
        base_generation, generation_balanced_six_ids,
        'native', 'balanced_shortcut_test_all_six_permutations',
    ),
    generation_metric(
        lora_generation, generation_balanced_six_ids,
        'lora', 'balanced_shortcut_test_all_six_permutations',
    ),
])
generation_summary.to_csv(
    ANALYSIS_DIR / 'generation_accuracy_summary.csv', index=False
)
display(generation_summary.round(4))

cell_stage('Creating paired confusion matrices and headline plot')
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
for axis, frame, title in (
    (axes[0], base_scene, 'Native Qwen3.5-27B'),
    (axes[1], lora_scene, 'Frozen LoRA'),
):
    matrix = confusion_matrix(
        frame['outcome'], frame['prediction'], labels=SIDES
    )
    image = axis.imshow(matrix, cmap='Blues')
    axis.set_xticks(range(3), SIDES)
    axis.set_yticks(range(3), SIDES)
    axis.set_xlabel('Predicted')
    axis.set_ylabel('True')
    axis.set_title(title)
    for row in range(3):
        for column in range(3):
            axis.text(
                column, row, str(matrix[row, column]),
                ha='center', va='center',
                color='white' if matrix[row, column] > matrix.max() / 2 else 'black',
            )
fig.suptitle('Held-out semantic confusion matrices')
fig.tight_layout()
confusion_path = FIGURE_DIR / 'candidate_confusion_matrices.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
plt.show()

headline_subsets = (
    'all_probe_test', 'balanced_shortcut_test',
    'all_weight_conflict', 'balanced_weight_conflict',
)
plot_frame = candidate_summary[
    candidate_summary['subset'].isin(headline_subsets)
].pivot(index='subset', columns='model', values='accuracy').reindex(headline_subsets)
fig, axis = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(plot_frame))
width = 0.36
axis.bar(x - width / 2, plot_frame['native'], width, label='Native')
axis.bar(x + width / 2, plot_frame['lora'], width, label='LoRA')
axis.set_xticks(x, plot_frame.index, rotation=15, ha='right')
axis.set_ylim(0, 1)
axis.set_ylabel('Scene-level accuracy')
axis.grid(axis='y', alpha=0.25)
axis.legend(frameon=False)
axis.set_title('Qwen3.5-27B held-out performance before and after LoRA')
fig.tight_layout()
headline_path = FIGURE_DIR / 'heldout_accuracy_comparison.png'
fig.savefig(headline_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figures:', confusion_path, headline_path, sep='\n- ')


In [ ]:
%%cell_progress 14/14 Finalize the no-training evaluation record
cell_stage('Writing reproducibility and completion records')

required_outputs = (
    OUTPUT_ROOT / 'test_only_dataset_manifest.json',
    EVALUATION_MANIFEST_PATH,
    OUTPUT_ROOT / 'native_candidate_records.jsonl',
    OUTPUT_ROOT / 'lora_candidate_records.jsonl',
    OUTPUT_ROOT / 'native_generation_records.jsonl',
    OUTPUT_ROOT / 'lora_generation_records.jsonl',
    ANALYSIS_DIR / 'candidate_accuracy_by_subset.csv',
    ANALYSIS_DIR / 'generation_accuracy_summary.csv',
    ANALYSIS_DIR / 'paired_candidate_effects.json',
    FIGURE_DIR / 'candidate_confusion_matrices.png',
    FIGURE_DIR / 'heldout_accuracy_comparison.png',
)
for required in required_outputs:
    assert Path(required).is_file(), required

completion = {
    'schema_version': 'visualtorque-postlora-evaluation-complete-1.0',
    'status': 'complete',
    'training_performed': False,
    'gradient_updates': 0,
    'allowed_model_input_split': 'probe_test',
    'forbidden_model_input_splits': ['probe_train', 'probe_dev'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_manifest_sha256': sha256_file(ADAPTER_MANIFEST_PATH),
    'evaluation_manifest_sha256': sha256_file(EVALUATION_MANIFEST_PATH),
    'outputs': {
        'candidate_summary': str(
            ANALYSIS_DIR / 'candidate_accuracy_by_subset.csv'
        ),
        'generation_summary': str(
            ANALYSIS_DIR / 'generation_accuracy_summary.csv'
        ),
        'paired_effects': str(
            ANALYSIS_DIR / 'paired_candidate_effects.json'
        ),
        'confusion_figure': str(
            FIGURE_DIR / 'candidate_confusion_matrices.png'
        ),
        'accuracy_figure': str(
            FIGURE_DIR / 'heldout_accuracy_comparison.png'
        ),
    },
    'interpretation_boundary': (
        'This evaluation measures held-out behavior. It does not prove a '
        'mechanistic change, and probe_test is no longer blind after inspection.'
    ),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', completion)

print('Evaluation complete:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
print('\nRecommended order after reviewing these tables:')
print('1. Freeze this adapter and all evaluation choices.')
print('2. If needed, create one new blind confirmatory dataset; do not retune.')
print('3. Compare native versus LoRA representations on train/dev discovery only.')
print('4. Freeze relocated sites, then run causal tests on held-out data.')
